# Notebook 2 — Create the Labels

## Load the ML table from 01_read_join.ipynb

In [1]:
import pandas as pd

ml_table = pd.read_csv("../artifacts/01_ml_table.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])

ml_table.shape

(99441, 21)

**Note:** Loaded 99,441 rows and 21 columns — matches the ML table saved 
in Notebook 1. Row count confirms no data was lost or duplicated during loading.

## Check missing delivery dates and order status

In [2]:
print(ml_table["order_delivered_customer_date"].isna().sum())
print(ml_table["order_status"].value_counts())

2965
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


**Note:** 2,965 rows are missing a delivery date. This matches orders 
that are not yet "delivered" (canceled, shipped, processing, etc.) — 
only 96,478 orders have status "delivered". These non-delivered orders 
cannot have a late/on-time label and will be excluded.

## Keep delivered orders only

In [3]:
delivered = ml_table[ml_table["order_status"] == "delivered"].copy()
delivered = delivered.dropna(subset=["order_delivered_customer_date"])

print(delivered.shape)

(96470, 21)


**Note:** After filtering, 96,470 delivered orders remain with a valid 
delivery date (out of 96,478 "delivered" orders — 8 had a missing date 
despite the status and were dropped). This is our clean base for building 
the label.

## Build the is_late label

In [4]:
delivered["is_late"] = (
    delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
).astype(int)

delivered["is_late"].value_counts(normalize=True)

is_late
0    0.918876
1    0.081124
Name: proportion, dtype: float64

**Note:** Clear class imbalance — only 8.1% of delivered orders are late, 
91.9% are on time. A model that always predicts "on time" would get ~92% 
accuracy while being useless. This means: (1) accuracy alone is not a 
good metric later, and (2) techniques like class weighting will be needed 
when training the model.

## Manually verify the label on a sample of real orders

In [5]:
delivered[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]].sample(5, random_state=42)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late
9505,c6a73b421eb3e92ce86dbfbbd3530a8f,2018-03-13 17:13:17,2018-03-19,0
31123,b132124ca9d69faf63989e08a5851151,2018-06-04 19:26:52,2018-06-08,0
27721,7e25a1c58e68fa94300358caad65b944,2017-09-08 17:21:38,2017-09-25,0
74022,8418eb39cd68b52032566797b8f1bd11,2017-12-12 21:13:48,2017-12-20,0
36187,fe1ec86f91f3b5b6bc46fc3e4b8262cd,2017-12-13 01:16:52,2017-12-15,0


**Note:** Manually verified 5 random orders — in every case, the actual 
delivery date is before the estimated date, and is_late = 0 as expected. 
The label logic is working correctly.

## Save the labeled table

In [6]:
delivered.to_csv("../artifacts/02_labeled_table.csv", index=False)
print("Saved:", delivered.shape)

Saved: (96470, 22)
